## Objective

The objective of the Gold layer is to transform Silver layer data into business-ready analytical tables using dimensional modeling techniques.

This layer creates:
- Dimension tables
- Fact tables
- Business metrics
- Star schema relationships

The Gold layer is optimized for reporting and Power BI analytics.


In [54]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

StatementMeta(, 55ecc426-21ac-4395-bea1-f4d01f285b53, 56, Finished, Available, Finished, False)

### Silver Tables

#### Read Silver Layer Data

Load cleaned and standardized Silver layer tables into Spark DataFrames for Gold layer transformation.

Source tables:
- customers
- products
- orders
- order_items
- inventory
- returns

In [55]:
df_customers = spark.read.format("delta").load("Tables/silver_customers")
df_inventory = spark.read.format("delta").load("Tables/silver_inventory")
df_order_items = spark.read.format("delta").load("Tables/silver_order_items")
df_orders = spark.read.format("delta").load("Tables/silver_orders")
df_products = spark.read.format("delta").load("Tables/silver_products")
df_returns = spark.read.format("delta").load("Tables/silver_returns")
df_locations = spark.read.format("delta").load("Tables/silver_locations")

StatementMeta(, 55ecc426-21ac-4395-bea1-f4d01f285b53, 57, Finished, Available, Finished, False)

## Build DimCustomer

The customer dimension contains descriptive customer attributes used for reporting and analytics.

This dimension supports:
- customer segmentation
- geographic analysis
- customer-level reporting
- regional analysis

A surrogate key is generated for warehouse modeling.

In [56]:
df_dim_location = (
    df_locations
    .select(
        "location_id",
        "city",
        "province",
        "country",
        "gta_region",
        "latitude",
        "longitude"
    )
    .dropDuplicates(["location_id"])
)

StatementMeta(, 55ecc426-21ac-4395-bea1-f4d01f285b53, 58, Finished, Available, Finished, False)

In [57]:
df_dim_customer = (
    df_customers.alias("c")
    .join(
        df_dim_location.select(
            "location_id",
            "city"
        ).alias("l"),
        col("c.city") == col("l.city"),
        "left"
    )
    .select(
        col("c.customer_id"),
        col("c.customer_name"),
        col("c.email"),
        col("c.address"),
        col("l.location_id"),
        col("c.postal_code"),
        col("c.customer_segment"),
        col("c.signup_date")
    )
    .dropDuplicates(["customer_id"])
)

window_customer = Window.orderBy("customer_id")

df_dim_customer = (
    df_dim_customer
    .withColumn("customer_key", row_number().over(window_customer))
)

df_dim_customer = df_dim_customer.select(
    "customer_key",
    "customer_id",
    "location_id",
    "customer_name",
    "email",
    "address",
    "postal_code",
    "customer_segment",
    "signup_date"
)

StatementMeta(, 55ecc426-21ac-4395-bea1-f4d01f285b53, 59, Finished, Available, Finished, False)

## Build DimProduct

The product dimension stores descriptive product information including:
- category
- subcategory
- brand
- pricing information

This dimension supports:
- product performance analysis
- category reporting
- profitability analysis

A surrogate product key is generated for dimensional modeling.

In [58]:
df_dim_product = (
    df_products
    .select(
        "product_id",
        "retailer_name",
        "category",
        "cost_price",
        "selling_price",
        "profit_margin",
        "is_active"
    )
    .dropDuplicates(["product_id"])
)

window_product = Window.orderBy("product_id")

df_dim_product = (
    df_dim_product
    .withColumn("product_key", row_number().over(window_product))
)

df_dim_product = df_dim_product.select(
    "product_key",
    "product_id",
    "retailer_name",
    "category",
    "cost_price",
    "selling_price",
    "profit_margin",
    "is_active"
)

StatementMeta(, 55ecc426-21ac-4395-bea1-f4d01f285b53, 60, Finished, Available, Finished, False)

## Build DimWarehouse

The warehouse dimension stores warehouse location information derived from inventory data.

This dimension supports:
- inventory tracking
- warehouse visibility
- operational reporting

A surrogate warehouse key is generated for analytical relationships.

In [59]:
df_dim_warehouse = (
    df_inventory
    .select("warehouse_location")
    .dropDuplicates(["warehouse_location"])
)

window_warehouse = Window.orderBy("warehouse_location")

df_dim_warehouse = (
    df_dim_warehouse
    .withColumn("warehouse_key", row_number().over(window_warehouse))
)

df_dim_warehouse = df_dim_warehouse.select(
    "warehouse_key",
    "warehouse_location"
)

StatementMeta(, 55ecc426-21ac-4395-bea1-f4d01f285b53, 61, Finished, Available, Finished, False)

## Build DimDate

The date dimension enables time-based analysis across sales and returns data.

This dimension includes:
- year
- month
- quarter
- day
- day name

The date dimension supports:
- trend analysis
- monthly reporting
- quarterly reporting
- time intelligence calculations

In [60]:
df_date_range = (
    df_orders
    .select(
        to_date(min("order_date")).alias("start_date"),
        lit("2026-01-31").cast("date").alias("end_date")
    )
)

df_dim_date = (
    df_date_range
    .select(
        explode(sequence(col("start_date"), col("end_date"))).alias("date")
    )
    .withColumn("date_key", date_format(col("date"), "yyyyMMdd").cast("int"))
    .withColumn("year", year(col("date")))
    .withColumn("month", month(col("date")))
    .withColumn("month_name", date_format(col("date"), "MMMM"))
    .withColumn("quarter", quarter(col("date")))
    .withColumn("day", dayofmonth(col("date")))
    .withColumn("day_name", date_format(col("date"), "EEEE"))
)

df_dim_date = df_dim_date.select(
    "date_key",
    "date",
    "year",
    "month",
    "month_name",
    "quarter",
    "day",
    "day_name"
)

display(df_dim_date)

StatementMeta(, 55ecc426-21ac-4395-bea1-f4d01f285b53, 62, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7c0d95ff-445c-421b-8a79-20198b232439)

## Build Fact Sales

The sales fact table combines transactional order data with product-level sales information.

Source tables:
- orders
- order_items

This table stores measurable business metrics including:
- gross sales
- total cost
- profit
- quantity sold

Dimension keys are integrated to support star schema relationships.

join orders and order items

In [61]:
df_fact_sales_base = (
    df_orders.alias("o")
    .join(
        df_order_items.alias("oi"),
        col("o.order_id") == col("oi.order_id"),
        "inner"
    )
)

StatementMeta(, 55ecc426-21ac-4395-bea1-f4d01f285b53, 63, Finished, Available, Finished, False)

Add Customer and Product Keys

Customer and product surrogate keys are integrated into the sales fact table to establish dimensional relationships.

In [62]:
df_fact_sales_keys = (
    df_fact_sales_base
    .join(
        df_dim_customer.select("customer_id", "customer_key"),
        "customer_id",
        "left"
    )
    .join(
        df_dim_product.select("product_id", "product_key"),
        "product_id",
        "left"
    )
)

StatementMeta(, 55ecc426-21ac-4395-bea1-f4d01f285b53, 64, Finished, Available, Finished, False)

join returns

In [63]:
df_returns_for_sales = (
    df_returns
    .select(
        "order_id",
        "return_date",
        "return_reason",
        "refund_amount"
    )
    .dropDuplicates(["order_id"])
)

df_fact_sales_with_returns = (
    df_fact_sales_keys
    .join(
        df_returns_for_sales,
        "order_id",
        "left"
    )
)

StatementMeta(, 55ecc426-21ac-4395-bea1-f4d01f285b53, 65, Finished, Available, Finished, False)

fact sales


In [64]:
df_fact_sales = (
    df_fact_sales_with_returns
    .select(
        col("oi.order_item_id").alias("order_item_id"),
        col("o.order_id").alias("order_id"),
        col("customer_key"),
        col("product_key"),
        col("o.customer_id").alias("customer_id"),
        col("oi.product_id").alias("product_id"),
        col("o.order_date").alias("order_date"),
        date_format(col("o.order_date"), "yyyyMMdd").cast("int").alias("date_key"),
        col("o.sales_channel").alias("sales_channel"),
        col("o.payment_method").alias("payment_method"),
        col("o.order_status").alias("order_status"),
        col("oi.quantity").alias("quantity"),
        col("oi.unit_price").alias("unit_price"),
        col("oi.unit_cost").alias("unit_cost"),
        col("oi.discount_rate").alias("discount_rate"),
        col("oi.discount_amount").alias("discount_amount"),
        col("oi.line_total").alias("gross_sales_amount"),
        col("oi.line_cost").alias("cost_amount"),
        col("oi.line_profit").alias("gross_profit_amount"),
        col("return_date"),
        col("return_reason"),
        coalesce(col("refund_amount"), lit(0)).alias("refund_amount"),
        col("o.order_year").alias("order_year"),
        col("o.order_month").alias("order_month")
    )
)

StatementMeta(, 55ecc426-21ac-4395-bea1-f4d01f285b53, 66, Finished, Available, Finished, False)

Net sales/profit

In [65]:
df_fact_sales = (
    df_fact_sales
    .withColumn(
        "net_sales_amount",
        col("gross_sales_amount") - col("refund_amount")
    )
    .withColumn(
        "net_profit_amount",
        when(col("refund_amount") > 0, lit(0))
        .otherwise(col("gross_profit_amount"))
    )
)

display(df_fact_sales.limit(5))

StatementMeta(, 55ecc426-21ac-4395-bea1-f4d01f285b53, 67, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 30abcd70-ae8f-40c0-bd7a-dab88cd59028)

## Fact Inventory

The inventory fact table tracks warehouse inventory activity and stock levels.

This table supports:
- stock monitoring
- inventory visibility
- warehouse inventory analysis
- replenishment reporting

###### Add Product and Warehouse Keys

Product and warehouse surrogate keys are integrated into the inventory fact table to support dimensional relationships.

In [66]:
df_fact_inventory = (
    df_inventory
    .join(
        df_dim_product.select("product_id", "product_key"),
        "product_id",
        "left"
    )
    .join(
        df_dim_warehouse.select("warehouse_location", "warehouse_key"),
        "warehouse_location",
        "left"
    )
)

StatementMeta(, 55ecc426-21ac-4395-bea1-f4d01f285b53, 68, Finished, Available, Finished, False)

##### Create Inventory Surrogate Key

A unique surrogate key is generated for inventory records.

In [67]:
window_inventory = Window.orderBy("inventory_id")

df_fact_inventory = (
    df_fact_inventory
    .withColumn("inventory_key", row_number().over(window_inventory))
)

StatementMeta(, 55ecc426-21ac-4395-bea1-f4d01f285b53, 69, Finished, Available, Finished, False)

In [68]:
df_fact_inventory = df_fact_inventory.select(
    "inventory_key",
    "inventory_id",
    "product_key",
    "warehouse_key",
    "product_id",
    "warehouse_location",
    "snapshot_date",
    "units_received",
    "units_sold",
    "ending_stock",
    "stock_status",
    "snapshot_year",
    "snapshot_month"
)

display(df_fact_inventory.limit(5))

StatementMeta(, 55ecc426-21ac-4395-bea1-f4d01f285b53, 70, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 91f3adb6-9994-458e-810e-bd9b6621e39e)

##### Perform Data Validation Checks

Validation checks are performed to ensure:
- successful row creation
- valid dimensional relationships
- successful surrogate key generation
- no missing dimension keys

In [69]:
print("Dim Customer:", df_dim_customer.count())
print("Dim Product:", df_dim_product.count())
print("Dim Warehouse:", df_dim_warehouse.count())
print("Dim Date:", df_dim_date.count())
print("Dim Location:", df_dim_location.count())
print("Fact Sales:", df_fact_sales.count())
print("Fact Inventory:", df_fact_inventory.count())

StatementMeta(, 55ecc426-21ac-4395-bea1-f4d01f285b53, 71, Finished, Available, Finished, False)

Dim Customer: 200
Dim Product: 50
Dim Warehouse: 4
Dim Date: 396
Dim Location: 10
Fact Sales: 60274
Fact Inventory: 600


##### Write Gold Tables

 Save Gold Layer Tables

The final Gold layer tables are written to the Lakehouse as Delta tables.

Gold tables created:
- gold_dim_customer
- gold_dim_product
- gold_dim_warehouse
- gold_dim_date
- gold_fact_sales
- gold_fact_inventory
- gold_fact_returns

In [70]:
print("Missing customer keys in sales:", df_fact_sales.filter(col("customer_key").isNull()).count())
print("Missing product keys in sales:", df_fact_sales.filter(col("product_key").isNull()).count())

print("Missing product keys in inventory:", df_fact_inventory.filter(col("product_key").isNull()).count())
print("Missing warehouse keys in inventory:", df_fact_inventory.filter(col("warehouse_key").isNull()).count())


StatementMeta(, 55ecc426-21ac-4395-bea1-f4d01f285b53, 72, Finished, Available, Finished, False)

Missing customer keys in sales: 0
Missing product keys in sales: 0
Missing product keys in inventory: 0
Missing warehouse keys in inventory: 0


In [71]:
df_dim_customer.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_dim_customer")

df_dim_product.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_dim_product")

df_dim_warehouse.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_dim_warehouse")

df_dim_date.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_dim_date")

df_fact_sales.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_fact_sales")

df_fact_inventory.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_fact_inventory")

df_dim_location.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_dim_location")

StatementMeta(, 55ecc426-21ac-4395-bea1-f4d01f285b53, 73, Finished, Available, Finished, False)

#### Gold Layer Complete

The Gold layer implements a dimensional star schema optimized for business intelligence and reporting.

The resulting analytical model supports:
- sales analytics
- inventory analysis
- returns reporting
- customer analysis
- product performance analysis
- Power BI dashboarding

## GOLD LAYER OPTIMIZATION
##### Optimize Gold tables for faster Power BI reporting

In [72]:
from pyspark.sql.functions import year, month

df_fact_sales = df_fact_sales \
    .withColumn("order_year", year("order_date")) \
    .withColumn("order_month", month("order_date"))

df_fact_inventory = df_fact_inventory \
    .withColumn("inventory_year", year("snapshot_date")) \
    .withColumn("inventory_month", month("snapshot_date"))


StatementMeta(, 55ecc426-21ac-4395-bea1-f4d01f285b53, 74, Finished, Available, Finished, False)

In [73]:
df_fact_sales.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("order_year", "order_month") \
    .save("Tables/gold/fact_sales")

df_fact_inventory.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("inventory_year", "inventory_month") \
    .save("Tables/gold/fact_inventory")


StatementMeta(, 55ecc426-21ac-4395-bea1-f4d01f285b53, 75, Finished, Available, Finished, False)

In [74]:
gold_tables = [
    "gold_dim_customer",
    "gold_dim_product",
    "gold_dim_warehouse",
    "gold_dim_date",
    "gold_fact_sales",
    "gold_fact_inventory",
    "gold_dim_location"
]

for table in gold_tables:
    spark.sql(f"OPTIMIZE {table}")

print("Gold layer optimization completed successfully.")

StatementMeta(, 55ecc426-21ac-4395-bea1-f4d01f285b53, 76, Finished, Available, Finished, False)

Gold layer optimization completed successfully.


In [75]:
df_fact_sales.printSchema()
df_fact_inventory.printSchema()

StatementMeta(, 55ecc426-21ac-4395-bea1-f4d01f285b53, 77, Finished, Available, Finished, False)

root
 |-- order_item_id: long (nullable = true)
 |-- order_id: long (nullable = true)
 |-- customer_key: integer (nullable = true)
 |-- product_key: integer (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- product_id: long (nullable = true)
 |-- order_date: timestamp (nullable = true)
 |-- date_key: integer (nullable = true)
 |-- sales_channel: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- quantity: long (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- unit_cost: double (nullable = true)
 |-- discount_rate: double (nullable = true)
 |-- discount_amount: double (nullable = true)
 |-- gross_sales_amount: double (nullable = true)
 |-- cost_amount: double (nullable = true)
 |-- gross_profit_amount: double (nullable = true)
 |-- return_date: timestamp (nullable = true)
 |-- return_reason: string (nullable = true)
 |-- refund_amount: double (nullable = false)
 |-- order_year: integer (nul

In [76]:
# Data quality checks for Gold layer

print("Gold layer validation started")

# Check row counts

print("Sales rows:", df_fact_sales.count())
print("Inventory rows:", df_fact_inventory.count())

# Check duplicate primary/business keys

print(
    "Duplicate order_item_id:",
    df_fact_sales.groupBy("order_item_id")
    .count()
    .filter("count > 1")
    .count()
)

print(
    "Duplicate inventory_key:",
    df_fact_inventory.groupBy("inventory_key")
    .count()
    .filter("count > 1")
    .count()
)

# Check null foreign keys

print(
    "Sales missing customer_key:",
    df_fact_sales.filter("customer_key IS NULL").count()
)

print(
    "Sales missing product_key:",
    df_fact_sales.filter("product_key IS NULL").count()
)

print(
    "Inventory missing product_key:",
    df_fact_inventory.filter("product_key IS NULL").count()
)

print(
    "Inventory missing warehouse_key:",
    df_fact_inventory.filter("warehouse_key IS NULL").count()
)

# Check negative values

print(
    "Negative sales quantity:",
    df_fact_sales.filter("quantity < 0").count()
)

print(
    "Negative gross sales:",
    df_fact_sales.filter("gross_sales_amount < 0").count()
)

print(
    "Negative gross profit:",
    df_fact_sales.filter("gross_profit_amount < 0").count()
)

print(
    "Negative ending stock:",
    df_fact_inventory.filter("ending_stock < 0").count()
)

print("Gold layer validation completed")

StatementMeta(, 55ecc426-21ac-4395-bea1-f4d01f285b53, 78, Finished, Available, Finished, False)

Gold layer validation started
Sales rows: 60274
Inventory rows: 600
Duplicate order_item_id: 0
Duplicate inventory_key: 0
Sales missing customer_key: 0
Sales missing product_key: 0
Inventory missing product_key: 0
Inventory missing warehouse_key: 0
Negative sales quantity: 0
Negative gross sales: 0
Negative gross profit: 888
Negative ending stock: 0
Gold layer validation completed


In [77]:
validation_results = [

    Row(
        check_name="sales_row_count",
        check_value=df_fact_sales.count()
    ),

    Row(
        check_name="inventory_row_count",
        check_value=df_fact_inventory.count()
    ),

    Row(
        check_name="duplicate_order_item_id",
        check_value=df_fact_sales
            .groupBy("order_item_id")
            .count()
            .filter("count > 1")
            .count()
    ),

    Row(
        check_name="duplicate_inventory_key",
        check_value=df_fact_inventory
            .groupBy("inventory_key")
            .count()
            .filter("count > 1")
            .count()
    ),

    Row(
        check_name="sales_missing_customer_key",
        check_value=df_fact_sales
            .filter("customer_key IS NULL")
            .count()
    ),

    Row(
        check_name="sales_missing_product_key",
        check_value=df_fact_sales
            .filter("product_key IS NULL")
            .count()
    ),

    Row(
        check_name="inventory_missing_product_key",
        check_value=df_fact_inventory
            .filter("product_key IS NULL")
            .count()
    ),

    Row(
        check_name="inventory_missing_warehouse_key",
        check_value=df_fact_inventory
            .filter("warehouse_key IS NULL")
            .count()
    ),

    Row(
        check_name="negative_sales_quantity",
        check_value=df_fact_sales
            .filter("quantity < 0")
            .count()
    ),

    Row(
        check_name="negative_gross_sales",
        check_value=df_fact_sales
            .filter("gross_sales_amount < 0")
            .count()
    ),

    Row(
        check_name="negative_refund_amount",
        check_value=df_fact_sales
            .filter("refund_amount < 0")
            .count()
    ),

    Row(
        check_name="negative_ending_stock",
        check_value=df_fact_inventory
            .filter("ending_stock < 0")
            .count()
    )

]

df_validation_results = spark.createDataFrame(validation_results)

display(df_validation_results)

StatementMeta(, 55ecc426-21ac-4395-bea1-f4d01f285b53, 79, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d2c0e620-6374-4e4d-acc4-e1e1b6814eb6)

In [78]:
df_validation_results.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_validation_results")

StatementMeta(, 55ecc426-21ac-4395-bea1-f4d01f285b53, 80, Finished, Available, Finished, False)